# LONDON CSV

In [46]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

### Leer el csv y sacar por pantalla las cinco primeras filas.

In [47]:
df = pd.read_csv("data/london_airbnb.csv")
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,11551,Arty and Bright London Apartment in Zone 2,43039,Adriano,NaN,Lambeth,51.46225,-0.11732,Entire home/apt,88,3,185,2019-09-15,1.58,2,336
1,13913,Holiday London DB Room Let-on going,54730,Alina,NaN,Islington,51.56802,-0.11121,Private room,65,1,19,2019-10-07,0.17,2,365
2,90700,Sunny Notting Hill flat & terrace,491286,Chil,NaN,Kensington and Chelsea,51.51074,-0.19853,Entire home/apt,105,2,339,2019-07-30,3.33,2,268
3,15400,Bright Chelsea Apartment. Chelsea!,60302,Philippa,NaN,Kensington and Chelsea,51.48796,-0.16898,Entire home/apt,100,30,88,2019-09-23,0.73,1,158
4,92399,"MODERN SELF CONTAINED ARCHITECT FLATLET, ISLIN...",497366,Andrea & Mark,NaN,Islington,51.55071,-0.08547,Private room,77,1,207,2019-10-21,2.04,2,336


### Descripción del conjunto de datos, estándar

In [48]:
df.describe(include='all')

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
count,8.506800e+04,85042,8.506800e+04,85056,0.0,85068,85068.000000,85068.000000,85068,85068.000000,85068.000000,85068.000000,65062,65062.000000,85068.000000,85068.000000
unique,NaN,82446,NaN,14573,NaN,33,NaN,NaN,4,NaN,NaN,NaN,1818,NaN,NaN,NaN
top,NaN,Double room,NaN,Veeve,NaN,Westminster,NaN,NaN,Entire home/apt,NaN,NaN,NaN,2019-11-03,NaN,NaN,NaN
freq,NaN,37,NaN,1235,NaN,9588,NaN,NaN,47445,NaN,NaN,NaN,1688,NaN,NaN,NaN
mean,2.319997e+07,NaN,8.514442e+07,NaN,NaN,NaN,51.509798,-0.128343,NaN,122.336766,4.148105,17.471152,NaN,1.201995,22.151150,118.471987
std,1.132475e+07,NaN,8.603532e+07,NaN,NaN,NaN,0.046341,0.093034,NaN,220.749123,16.681720,36.789578,NaN,1.402728,110.654631,134.840097
min,1.155100e+04,NaN,2.697000e+03,NaN,NaN,NaN,51.294790,-0.496680,NaN,0.000000,1.000000,0.000000,NaN,0.010000,1.000000,0.000000
25%,1.441315e+07,NaN,1.529856e+07,NaN,NaN,NaN,51.485610,-0.188412,NaN,47.000000,1.000000,1.000000,NaN,0.250000,1.000000,0.000000
50%,2.363201e+07,NaN,4.762387e+07,NaN,NaN,NaN,51.514015,-0.125725,NaN,84.000000,2.000000,4.000000,NaN,0.730000,1.000000,58.000000
75%,3.349281e+07,NaN,1.415435e+08,NaN,NaN,NaN,51.537580,-0.070127,NaN,140.000000,3.000000,17.000000,NaN,1.630000,5.000000,244.000000


### Información sobre el tipo de datos de cada feature

In [49]:
print("--- Información de tipos de datos ---")
df.info()

--- Información de tipos de datos ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85068 entries, 0 to 85067
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              85068 non-null  int64  
 1   name                            85042 non-null  object 
 2   host_id                         85068 non-null  int64  
 3   host_name                       85056 non-null  object 
 4   neighbourhood_group             0 non-null      float64
 5   neighbourhood                   85068 non-null  object 
 6   latitude                        85068 non-null  float64
 7   longitude                       85068 non-null  float64
 8   room_type                       85068 non-null  object 
 9   price                           85068 non-null  int64  
 10  minimum_nights                  85068 non-null  int64  
 11  number_of_reviews               85068 non-null  int64  

dataset de AirBnB con 19,618 propiedades y 15 columnas

### Contar los nulos por variable

In [50]:
print("--- Valores nulos por columna ---")
null_counts = df.isnull().sum()
print(null_counts)
if null_counts.sum() > 0:
    print("\nColumnas con valores nulos:")
    print(null_counts[null_counts > 0])
else:
    print("✓ No hay valores nulos en el dataset")

--- Valores nulos por columna ---
id                                    0
name                                 26
host_id                               0
host_name                            12
neighbourhood_group               85068
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       20006
reviews_per_month                 20006
calculated_host_listings_count        0
availability_365                      0
dtype: int64

Columnas con valores nulos:
name                      26
host_name                 12
neighbourhood_group    85068
last_review            20006
reviews_per_month      20006
dtype: int64


#### Buscar valores extraños. Ver los valores únicos en cada feature

In [51]:
unique_values = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_values
unique_vals = pd.DataFrame({
    'features': df.columns,
    'n_values': [df[col].unique() for col in df.columns]
})
unique_vals

,features,n_values
0,id,"[11551, 13913, 90700, 15400, 92399, 17402, 926..."
1,name,"[Arty and Bright London Apartment in Zone 2, H..."
2,host_id,"[43039, 54730, 491286, 60302, 497366, 67564, 4..."
3,host_name,"[Adriano, Alina, Chil, Philippa, Andrea & Mark..."
4,neighbourhood_group,[nan]
5,neighbourhood,"[Lambeth, Islington, Kensington and Chelsea, W..."
6,latitude,"[51.46225, 51.56802, 51.51074, 51.48796, 51.55..."
7,longitude,"[-0.11732, -0.11121, -0.19853, -0.16898, -0.08..."
8,room_type,"[Entire home/apt, Private room, Hotel room, Sh..."
9,price,"[88, 65, 105, 100, 77, 300, 42, 79, 180, 29, 1..."


### Obtener dataframe con features y sus valores únicos

In [52]:
unique_values_info = []
for col in df.columns:
    unique_vals = df[col].unique()
    unique_values_info.append({
        'features': col,
        'n_values': len(unique_vals),
        'values': str(list(unique_vals)[:10])  # Mostrar solo los primeros 10
    })

unique_df = pd.DataFrame(unique_values_info)
print("--- Valores únicos por feature ---")
unique_df[['features', 'n_values']]

--- Valores únicos por feature ---


,features,n_values
0,id,85068
1,name,82447
2,host_id,53476
3,host_name,14574
4,neighbourhood_group,1
5,neighbourhood,33
6,latitude,20713
7,longitude,32079
8,room_type,4
9,price,881


#### Tratar aquellos valores que entendamos que sean nulos

In [53]:
print("=== VERIFICANDO VALORES EXTRAÑOS ===")
strange_values = ['?', 'N/A', 'n/a', 'NA', 'na', 'Unknown', 'unknown', '', ' ', 'NULL', 'null']
strange_values_found = False

for col in df.columns:
    if df[col].dtype == 'object':
        unique_vals = df[col].unique()
        for strange_val in strange_values:
            if strange_val in unique_vals:
                count_strange = (df[col] == strange_val).sum()
                if count_strange > 0:
                    print(f"'{col}' contiene {count_strange} valores '{strange_val}' (posibles nulos)")
                    strange_values_found = True

if not strange_values_found:
    print("✓ No se encontraron valores extraños aparentes")
else:
    print("\n--- Convirtiendo valores extraños a NaN ---")
    # Convertir valores extraños a NaN
    for col in df.columns:
        if df[col].dtype == 'object':
            for strange_val in strange_values:
                df[col] = df[col].replace(strange_val, np.nan)

print("\n=== TRATAMIENTO DE VALORES FALTANTES ===")
original_shape = df.shape
print(f"Shape original del dataset: {original_shape}")

=== VERIFICANDO VALORES EXTRAÑOS ===
✓ No se encontraron valores extraños aparentes

=== TRATAMIENTO DE VALORES FALTANTES ===
Shape original del dataset: (85068, 16)


#### Estrategias específicas por columna según tu dataset


In [54]:
tratamiento_por_columna = {
    'name': 'imputar_generic',  # 26 nulos - imputar con nombre genérico
    'host_name': 'imputar_unknown',  # 12 nulos - imputar con 'Unknown Host'
    'neighbourhood_group': 'eliminar_columna',  # 85068 nulos (100%) - eliminar columna
    'last_review': 'mantener_null',  # 20006 nulos - mantener (indica sin reviews)
    'reviews_per_month': 'imputar_cero'  # 20006 nulos - coherente con sin reviews
}

#### Mirad cuántos valores hay en cada feature, ¿Todas las features aportan información? Si alguna no aporta información, eliminadla

In [55]:
# Eliminar features que no aportan información (solo un valor único)
print("--- Verificando features informativas ---")
single_value_cols = []
for col in df.columns:
    unique_count = df[col].nunique()
    if unique_count <= 1:
        single_value_cols.append(col)
        print(f"⚠️  '{col}' tiene solo {unique_count} valor único")

if single_value_cols:
    df = df.drop(columns=single_value_cols)
    print(f"✓ Eliminadas {len(single_value_cols)} columnas sin información")
else:
    print("✓ Todas las features aportan información")

print(f"Forma final del dataset: {df.shape}")

--- Verificando features informativas ---
⚠️  'neighbourhood_group' tiene solo 0 valor único
✓ Eliminadas 1 columnas sin información
Forma final del dataset: (85068, 15)


#### Separar entre variables predictoras y variables a predecir

In [56]:
target_col = 'price' if 'price' in df.columns else df.columns[0]
y = df[target_col]
X = df.drop(columns=[target_col])

print(f"Variable objetivo: {target_col}")
print(f"Variables predictoras: {X.shape[1]} features")
print(f"\n--- Distribución de la variable objetivo ---")
print(y.value_counts())
print(f"\nPorcentajes:")
print(y.value_counts(normalize=True) * 100)

Variable objetivo: price
Variables predictoras: 14 features

--- Distribución de la variable objetivo ---
price
50      3116
100     3066
40      2750
35      2450
120     2393
        ... 
615        1
1477       1
820        1
879        1
4          1
Name: count, Length: 881, dtype: int64

Porcentajes:
price
50      3.662952
100     3.604175
40      3.232708
35      2.880049
120     2.813044
          ...   
615     0.001176
1477    0.001176
820     0.001176
879     0.001176
4       0.001176
Name: proportion, Length: 881, dtype: float64


In [57]:
# Aplicar tratamientos
filas_a_eliminar = set()
columnas_a_eliminar = []

for col in df.columns:
    if df[col].isnull().sum() > 0:
        null_count = df[col].isnull().sum()
        print(f"\nTratando '{col}' ({null_count} valores nulos)...")
        
        if col in tratamiento_por_columna:
            estrategia = tratamiento_por_columna[col]
            
            if estrategia == 'eliminar_fila':
                # Marcar filas para eliminación
                filas_nulas = df[df[col].isnull()].index
                filas_a_eliminar.update(filas_nulas)
                print(f"  → Marcadas {len(filas_nulas)} filas para eliminación")
                
            elif estrategia == 'eliminar_columna':
                # Marcar columna para eliminación
                columnas_a_eliminar.append(col)
                print(f"  → Marcada columna para eliminación (100% nulos)")
                
            elif estrategia == 'imputar_unknown':
                df[col] = df[col].fillna('Unknown Host')
                print(f"  → Imputado con 'Unknown Host'")
                
            elif estrategia == 'imputar_generic':
                df[col] = df[col].fillna('Unnamed Listing')
                print(f"  → Imputado con 'Unnamed Listing'")
                    
            elif estrategia == 'imputar_cero':
                df[col] = df[col].fillna(0)
                print(f"  → Imputado con 0")
                
            elif estrategia == 'mantener_null':
                print(f"  → Manteniendo valores nulos (representan ausencia de reviews)")
                
        else:
            # Estrategia por defecto según el tipo de dato
            if df[col].dtype in ['int64', 'float64']:
                # Para columnas numéricas, usar la mediana
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"  → Imputado con mediana: {median_val}")
            else:
                # Para columnas categóricas, usar la moda
                if len(df[col].mode()) > 0:
                    mode_val = df[col].mode()[0]
                    df[col] = df[col].fillna(mode_val)
                    print(f"  → Imputado con moda: {mode_val}")
                else:
                    df[col] = df[col].fillna('Unknown')
                    print(f"  → Imputado con 'Unknown'")
# Eliminar filas marcadas
if filas_a_eliminar:
    df = df.drop(index=filas_a_eliminar)
    print(f"\n✓ Eliminadas {len(filas_a_eliminar)} filas con valores nulos")
# Eliminar columnas marcadas
if columnas_a_eliminar:
    df = df.drop(columns=columnas_a_eliminar)
    print(f"✓ Eliminadas {len(columnas_a_eliminar)} columnas con 100% de valores nulos")
print(f"\nForma final del dataset: {df.shape}")
# Verificar si hay valores nulos restantes
if df.isnull().sum().sum() > 0:
    print("\n⚠️ Aún quedan valores nulos en el dataset:")
    print(df.isnull().sum()[df.isnull().sum() > 0])


Tratando 'name' (26 valores nulos)...
  → Imputado con 'Unnamed Listing'

Tratando 'host_name' (12 valores nulos)...
  → Imputado con 'Unknown Host'

Tratando 'last_review' (20006 valores nulos)...
  → Manteniendo valores nulos (representan ausencia de reviews)

Tratando 'reviews_per_month' (20006 valores nulos)...
  → Imputado con 0

Forma final del dataset: (85068, 15)

⚠️ Aún quedan valores nulos en el dataset:
last_review    20006
dtype: int64


In [58]:
# Verificación final
print(f"\n=== RESULTADO FINAL ===")
print(f"✓ Dataset shape: {original_shape} → {df.shape}")
print(f"✓ Filas eliminadas: {original_shape[0] - df.shape[0]}")

# Verificar si quedan valores nulos
remaining_nulls = df.isnull().sum().sum()
if remaining_nulls > 0:
    print(f"⚠️  Quedan {remaining_nulls} valores nulos:")
    null_counts_final = df.isnull().sum()
    for col, count in null_counts_final[null_counts_final > 0].items():
        print(f"   {col}: {count}")
else:
    print("✓ No quedan valores nulos en el dataset")

# Mostrar información del dataset tratado
print(f"\n=== INFORMACIÓN DEL DATASET FINAL ===")
print(df.info())


=== RESULTADO FINAL ===
✓ Dataset shape: (85068, 16) → (85068, 15)
✓ Filas eliminadas: 0
⚠️  Quedan 20006 valores nulos:
   last_review: 20006

=== INFORMACIÓN DEL DATASET FINAL ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85068 entries, 0 to 85067
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              85068 non-null  int64  
 1   name                            85068 non-null  object 
 2   host_id                         85068 non-null  int64  
 3   host_name                       85068 non-null  object 
 4   neighbourhood                   85068 non-null  object 
 5   latitude                        85068 non-null  float64
 6   longitude                       85068 non-null  float64
 7   room_type                       85068 non-null  object 
 8   price                           85068 non-null  int64  
 9   minimum_nights                  